# 用 SAT 解数独 ― 把“恰好一个为真”写成 CNF

## 这一章要学什么

数独这种东西，我们打算交给 SAT（可满足性问题）求解器去解。从一道小的推理谜题起步，把“问题怎么翻成布尔公式”这条流水线走一遍，最后一直走到数独。

- SAT 和 CNF（合取范式）到底在说什么
- 把析取范式（DNF）换成 CNF 的路子（SymPy 一条，手写一条）
- “一组里只有一个为 True”这类约束该怎么下笔
- 用 `pysat` 解数独，用 `assumptions` 把初始盘面递进去

## 可满足性问题（SAT）

**可满足性问题**（SATisfiability problem，SAT）问的是这么一件事：给定一个布尔公式，能不能找出一组变量的取值，让它为真？

SAT 是历史上第一个被证明为 **NP 完全**的问题，计算机科学的基础理论、算法、人工智能、硬件设计，一大堆领域都要跟它打交道。可真正让人乐意用它的理由是另一个：实用的 SAT 求解器跑得极快。于是“把想解的问题先翻译成 SAT”就成了很常见的手法。

这一章我们用 `pysat` 来解数独。不过在动数独之前，先拿一道小推理谜题，把布尔公式和 CNF 的写法过一遍。

> **推理谜题**
>
> 4 名嫌疑人作了这样的供述。
>
> - 甲：“我不是犯人”
> - 乙：“丁是犯人”
> - 丙：“乙是犯人”
> - 丁：“乙想陷害我”
>
> 4 个人当中，说真话的只有 1 个。犯人是谁？

甲、乙、丙、丁四个人，各自“是不是犯人”，各用一个布尔变量来表示：**A**、**B**、**C**、**D**。`~A` 就是“甲不是犯人”。

四条供述，写成布尔公式是这样。

| 嫌疑人 | 供述 | 布尔公式 |
|:--|:--|:--|
| 甲 | 我不是犯人 | S1 = ~A |
| 乙 | 丁是犯人 | S2 = D |
| 丙 | 乙是犯人 | S3 = B |
| 丁 | 乙想陷害我 | S4 = ~D |

说真话的只有一个，所以 S1～S4 里头**恰好有一个为真**。

先把“恰好一个为真”老老实实写出来看看（`&` 是逻辑与，`|` 是逻辑或）。

```
 S1 & ~S2 & ~S3 & ~S4 |
~S1 &  S2 & ~S3 & ~S4 |
~S1 & ~S2 &  S3 & ~S4 |
~S1 & ~S2 & ~S3 &  S4
```

这一串是什么？是用 OR 串起来的若干块 AND。这种形态叫**析取范式**（DNF，Disjunctive Normal Form）。

可 SAT 求解器只认**合取范式**（CNF，Conjunctive Normal Form），也就是“把用 OR 连起来的子句，拿 AND 排成一列”。既然这样，就换个写法来写“恰好一个为真”。

1. 不许两个以上同时为真 → `~(Si & Sj)`
2. 也不许全都为假 → `~( ~S1 & ~S2 & ~S3 & ~S4 )`

```
~(S1 & S2) & ~(S1 & S3) & ~(S1 & S4) &
~(S2 & S3) & ~(S2 & S4) & ~(S3 & S4) &
~(~S1 & ~S2 & ~S3 & ~S4)
```

再拿德摩根定律（`~(X & Y) = ~X | ~Y`）变一下形，CNF 就出来了。

**转换结果**（把 S1 = ~A, S2 = D, S3 = B, S4 = ~D 代进去）

```
~S1 | ~S2   →   A  | ~D
~S1 | ~S3   →   A  | ~B
~S1 | ~S4   →   A  |  D
~S2 | ~S3   →  ~D  | ~B
~S2 | ~S4   →  ~D  |  D     ← 恒为真（什么也约束不了）
~S3 | ~S4   →  ~B  |  D
 S1 |  S2 | S3 | S4  →  ~A | D | B | ~D
```

`~D | D` 永远为真，所以这一条子句什么也约束不了。留着也不碍事，就原样放进去了。

## 让 SymPy 从 DNF 走到 CNF

手算这种变形太容易出错，还是交给 SymPy。

In [1]:
from sympy import symbols
from sympy.logic.boolalg import to_cnf

A, B, C, D = symbols("A:D")

# 每人的供述
S1 = ~A
S2 = D
S3 = B
S4 = ~D

# 只有 1 个人说真话（DNF）
dnf = ((S1 & ~S2 & ~S3 & ~S4) |
       (~S1 & S2 & ~S3 & ~S4) |
       (~S1 & ~S2 & S3 & ~S4) |
       (~S1 & ~S2 & ~S3 & S4))

# simplify=True：先化简式子再返回
print("CNF:", to_cnf(dnf, simplify=True))

# simplify=False：按分配律直接展开
cnf_raw = to_cnf(dnf, simplify=False)
print("simplify=False 时会有", len(cnf_raw.args), "个子句")

CNF: A & ~B
simplify=False 时会有 41 个子句


`simplify=True` 打开以后，SymPy 会先把式子化简再交回来。结果是 `A & ~B`，说白了就是“甲是犯人，乙不是犯人”。

换成 `simplify=False`，它就照分配律一路展开，子句数能涨到几十个。`A | ~A` 这种恒真的子句也不会被顺手删掉。数独这种规模的问题上，这份膨胀会一分不少地变成求解时间。

两条路得到的 CNF 长得不一样，答案却该是同一个。

In [2]:
from sympy.logic.inference import satisfiable

print(satisfiable(to_cnf(dnf, simplify=True)))
print(satisfiable(cnf_raw))

{A: True, B: False}
{A: True, B: False, D: False}


`satisfiable()` 交回来的，是让式子为真的那组变量赋值。`A` 为真，所以**犯人是甲**。`C` 根本没在式子里露面，取真取假都行，也就是没有约束。

## 交给 pysat

CNF 可以直接写成嵌套列表。一个整数对应一个布尔变量：正数说这个变量为真，负数说它为假，也就是取否定。`1` 是 A，`-1` 就是 ~A。

上面那串 CNF，照抄成数值列表就行（`A=1, B=2, C=3, D=4`）。

1. 用 `append_formula()` 把 CNF 塞进求解器（调几次都行）
2. 用 `solve()` 求解
3. 用 `get_model()` 把解取出来（正数为真）

In [3]:
from pysat.solvers import Solver

sat = Solver()
problem = [[1, -4], [1, -2], [1, 4],            # ~S1 | ~S2 等
           [-4, -2], [-4, 4], [-2, 4],          # ~D | D 恒为真
           [-1, 4, 2, -4]]                      # S1 | S2 | S3 | S4

sat.append_formula(problem)
print(sat.solve())
print(sat.get_model())

True
[1, -2, -3, -4]


`[1, -2, -3, -4]` 说的是“A 真，B、C、D 假”。结论和 SymPy 那边一模一样。

## 数独

数独这个谜题，是在 9×9 的格子里填 1～9，并且要满足下面几条。

- 每一行里 1～9 各出现一次
- 每一列里 1～9 各出现一次
- 每个 3×3 区块里 1～9 各出现一次

![数独游戏](images/sudoku01.png)

黑色的数字是题目，灰色的数字是答案。

SAT 的变量只有 `True` / `False` 两个值，所以我们把“格子 (r, c) 里放数字 n”整件事做成一个布尔变量，用三维数组 `bools` 来装。

- 第 0 轴 … 行
- 第 1 轴 … 列
- 第 2 轴 … 数字（1～9）

比如 `bools[4, 1, 3]` 为真，读作“第 5 行第 2 列放数字 4”。数独从 1 开始数，数组下标却从 0 开始，这里差一位，别弄混。

`bools` 里装的是 SAT 的变量编号，编号从 1 开始。

In [4]:
import numpy as np

# 9×9×9 = 729 个变量（1〜729）
bools = np.arange(1, 9 * 9 * 9 + 1).reshape(9, 9, 9)

数独的约束，理一理其实就 4 类。

| # | 约束 | 分组方法 |
|---|---|---|
| 1 | 每个格子只放一个数字 | 把数字轴（第 2 轴）当成一组 |
| 2 | 每一行里同一个数字只有一个 | 按行和数字分组 |
| 3 | 每一列里同一个数字只有一个 | 按列和数字分组 |
| 4 | 每个区块里同一个数字只有一个 | 按区块和数字分组 |

四类说的其实是同一句话：**某一组里头，恰好只有一个为 True**。

## 用 CNF 写“恰好一个”

`A`、`B`、`C` 三个变量里只有一个为真，这条约束能拆成两个条件来看。

1. 至少有一个为真: $A | B | C$
2. 不许两个以上同时为真: $\neg(A \& B) \land \neg(A \& C) \land \neg(B \& C)$

第二个条件过一遍德摩根定律，就成了 $ (\neg A | \neg B) \land (\neg A | \neg C) \land (\neg B | \neg C) $。

$$
(A | B | C) \land (\neg A | \neg B) \land (\neg A | \neg C) \land (\neg B | \neg C)
$$

推广到 N 个元素：一条“把全部用 OR 串起来”的子句，加上 $N(N-1)/2$ 条“每个二元组都写一句 `~X | ~Y`”。二元组用 `itertools.combinations` 生成就行。

`get_conditions(bools)` 把 `bools` 的**最后一轴**当成一组，返回的就是这条约束。

1. 看最后一轴有几个元素，记为 `n`，把二元组全造出来
2. 给每一组造一条用 OR 串起来的子句（至少一个为真）
3. 给每个二元组造一条 `~X | ~Y` 子句（两个不许同时为真）

In [5]:
from itertools import combinations


def get_conditions(bools):
    """把最后一轴作为一组，生成“恰好一个为 True”的 CNF"""
    n = bools.shape[-1]
    pairs = np.array(list(combinations(range(n), 2)))

    conditions = []
    # 1. 至少一个为 True
    conditions.extend(bools.reshape(-1, n).tolist())
    # 2. 两个不能同时为 True
    conditions.extend((-bools[..., pairs].reshape(-1, 2)).tolist())
    return conditions

拿个小例子看看它怎么动。`[[1, 2, 3], [4, 5, 6]]` 的意思是“两组，每组三个变量”。

- `[1, 2, 3]`, `[4, 5, 6]` … 每一组里至少有一个为真
- `[-1, -2]`, `[-1, -3]`, `[-2, -3]` … 1～3 里不许有两个同时为真
- `[-4, -5]`, `[-4, -6]`, `[-5, -6]` … 4～6 也一样

In [6]:
get_conditions(np.array([[1, 2, 3], [4, 5, 6]]))

[[1, 2, 3],
 [4, 5, 6],
 [-1, -2],
 [-1, -3],
 [-2, -3],
 [-4, -5],
 [-4, -6],
 [-5, -6]]

## 把数独的约束造出来

`get_conditions()` 只认最后一轴，所以行、列、区块这几类约束，得先用 `swapaxes()` 把**想分组的轴挪到最后**，再递进去。

- 约束 1（每个格子一个）… 原样递（最后一轴本来就是数字）
- 约束 2（每行一个）… 列和数字换位
- 约束 3（每列一个）… 行和数字换位

In [7]:
c1 = get_conditions(bools)                     # 每个格子只有一个数字
c2 = get_conditions(np.swapaxes(bools, 1, 2))  # 每行中同一数字只有一个
c3 = get_conditions(np.swapaxes(bools, 0, 2))  # 每列中同一数字只有一个

约束 4（区块）要多绕一步。`bools` 的轴是 `(行, 列, 数字)`，得把行和列各自拆成“区块编号”和“区块内的位置”两段。

1. `reshape(3, 3, 3, 3, 9)` … `(区块行, 行内, 区块列, 列内, 数字)`
2. `swapaxes(1, 2)` … `(区块行, 区块列, 行内, 列内, 数字)`
3. `reshape(9, 9, 9)` … `(区块编号, 区块内的 9 个格子, 数字)`
4. `swapaxes(1, 2)` … `(区块编号, 数字, 区块内的 9 个格子)` ← 分组的轴落在最后

到这儿，“每个区块、每个数字，9 个格子里只有一个为真”就成立了。

In [8]:
# 按 (区块, 9 个格子, 数字) 的形状整理
blocks = np.swapaxes(bools.reshape(3, 3, 3, 3, 9), 1, 2).reshape(9, 9, 9)
c4 = get_conditions(np.swapaxes(blocks, 1, 2))   # 每个区块中同一数字只有一个

conditions = c1 + c2 + c3 + c4
print(len(conditions), "个子句")

11988 个子句


## 先解一个空盘面

什么提示都不给，先让它解一次。满足数独约束的答案会出来一个（没有提示，答案本来就有无数个）。

模型是一串变量编号，所以要重新摆成 `(9, 9, 9)`，再把为真的那些变量的数字取出来，也就是最后一轴的下标 + 1。

In [9]:
def format_solution(model, shape=(9, 9, 9)):
    """把模型（变量编号的列表）变成 9×9 的数字盘面"""
    grid = np.array(model).reshape(shape)
    return np.where(grid > 0)[2].reshape(9, 9) + 1


sat = Solver()
sat.append_formula(conditions)
print(sat.solve())
format_solution(sat.get_model())

True


array([[2, 4, 6, 1, 3, 5, 8, 7, 9],
       [1, 3, 5, 7, 8, 9, 2, 4, 6],
       [8, 7, 9, 2, 4, 6, 1, 3, 5],
       [5, 6, 3, 4, 1, 2, 7, 9, 8],
       [4, 1, 2, 9, 7, 8, 5, 6, 3],
       [7, 9, 8, 5, 6, 3, 4, 1, 2],
       [6, 5, 4, 3, 2, 1, 9, 8, 7],
       [3, 2, 1, 8, 9, 7, 6, 5, 4],
       [9, 8, 7, 6, 5, 4, 3, 2, 1]])

## 解一道真的题

题目用字符串给，`0` 表示空格子。非空的地方，只要追加一条“这个数字的变量为真”的约束，那格就固定成那个数字了。

In [10]:
sudoku_str = """
000000185
007030000
000021400
800000020
003905600
050000004
004860000
000040300
931000000"""

puzzle = np.array([[int(x) for x in line]
                   for line in sudoku_str.strip().split()])

# 对提示格加上“就是那个数字”的约束
r, c = np.where(puzzle != 0)
v = puzzle[r, c] - 1
hints = [[int(x)] for x in bools[r, c, v]]

sat = Solver()
sat.append_formula(conditions + hints)
sat.solve()
solution = format_solution(sat.get_model())
solution

array([[3, 6, 2, 7, 9, 4, 1, 8, 5],
       [4, 1, 7, 5, 3, 8, 2, 6, 9],
       [5, 9, 8, 6, 2, 1, 4, 3, 7],
       [8, 7, 9, 4, 1, 6, 5, 2, 3],
       [2, 4, 3, 9, 7, 5, 6, 1, 8],
       [1, 5, 6, 3, 8, 2, 7, 9, 4],
       [7, 2, 4, 8, 6, 3, 9, 5, 1],
       [6, 8, 5, 1, 4, 9, 3, 7, 2],
       [9, 3, 1, 2, 5, 7, 8, 4, 6]])

`hints` 里每一条子句都只含一个布尔变量，写成 `[x]`。它的意思就是“x 必须为真”。

## 验算

解出来的盘面，是不是真的满足数独的条件？得查一遍。

In [11]:
def check_sudoku(grid):
    """每行、每列、每个区块是否都是 1〜9 各一次"""
    target = set(range(1, 10))
    rows = [set(grid[i, :]) for i in range(9)]
    cols = [set(grid[:, j]) for j in range(9)]
    blocks = [set(grid[i:i+3, j:j+3].ravel())
              for i in range(0, 9, 3) for j in range(0, 9, 3)]
    return all(g == target for g in rows + cols + blocks)


print(check_sudoku(solution))

# 也确认与提示不矛盾
print(bool((puzzle[puzzle > 0] == solution[puzzle > 0]).all()))

True
True


## 用 assumptions 递提示

除了像 `hints` 那样追加“只含一个变量的子句”，也可以把想让它为真的那批变量，直接递给 `solve()` 的 `assumptions`。

`assumptions` 在求完解之后会被清掉，所以**同一个求解器可以反复用，一道接一道地解**。要连解很多题，走这条路更省事。

In [12]:
sat = Solver()
sat.append_formula(conditions)                     # 只放入数独的规则

sat.solve(assumptions=bools[r, c, v].tolist())     # 提示作为假设传入
solution2 = format_solution(sat.get_model())
print(check_sudoku(solution2))
solution2

True


array([[3, 6, 2, 7, 9, 4, 1, 8, 5],
       [4, 1, 7, 5, 3, 8, 2, 6, 9],
       [5, 9, 8, 6, 2, 1, 4, 3, 7],
       [8, 7, 9, 4, 1, 6, 5, 2, 3],
       [2, 4, 3, 9, 7, 5, 6, 1, 8],
       [1, 5, 6, 3, 8, 2, 7, 9, 4],
       [7, 2, 4, 8, 6, 3, 9, 5, 1],
       [6, 8, 5, 1, 4, 9, 3, 7, 2],
       [9, 3, 1, 2, 5, 7, 8, 4, 6]])

## 小结

- SAT 解的是“有没有一组赋值能让布尔公式为真”。它虽然是 NP 完全，实用的求解器却很快
- 求解器只吃 CNF（合取范式），DNF 没法直接递进去
- “恰好一个为真”拆成“至少一个”＋“所有二元组都写 `~X | ~Y`”，就变成 CNF 了
- 数独把“格子 × 数字”做成布尔变量，再排上格子、行、列、区块 4 类约束
- 用 `swapaxes()` 把想分组的轴挪到最后，同一个函数就能反复用
- 初始盘面既可以用 `hints`（单变量子句）给，也可以用 `assumptions` 给。要复用同一个求解器，就选 `assumptions`